# 04 · Teammate race pace

Qualifying shows one-lap speed; this compares teammates' **race pace**.

**Clean laps:** only laps FastF1 marks as accurate. That flag already requires no pit in/out lap, a real lap time, sector times that add up, and green or yellow track status only, so safety car, VSC and red-flag laps are out.

**Wet races** are excluded for now (choose the definition below).

**Model**, fitted separately for each race:

`lap_time ~ driver + compound + compound × tyre_age + lap_number`

It adjusts each lap for the tyre compound, how worn that tyre is (each compound degrading at its own rate) and lap number (fuel burn-off and track grip). The teammate gap is the difference between the two drivers' estimated pace with all of that held equal. By default a robust (Huber) regression is used, so laps lost in traffic or to mistakes count less.

`GapSE` is the standard error of each race's gap: a race gap smaller than about twice its SE could be noise.

Run notebook 03 first.

In [19]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.17 KiB | 400.00 KiB/s, done.
From https://github.com/khushi075/f1-race-sim
   c60008e..8fc0e27  main       -> origin/main
Updating c60008e..8fc0e27
Fast-forward
 f1sim/race_pace.py                    | 22 ++++++++++++++++++++--
 notebooks/04_teammate_race_pace.ipynb | 11 +++++++----
 2 files changed, 27 insertions(+), 6 deletions(-)
/content/f1-race-sim
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [20]:
# @title Settings and model fit
WET_RULE = "tyres"  # @param ["tyres_or_rain", "tyres"]
MIN_LAPS = 10  # @param {type:"integer"}
ROBUST = True  # @param {type:"boolean"}

import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest_tracing import load_race_ti
from f1sim.race_pace import (wet_races, clean_dry_laps, teammate_race_gaps,
                             race_season_summary)

laps = load_race_ti("laps", get_paths())
wet = wet_races(laps, WET_RULE)
clean = clean_dry_laps(laps, exclude_wet=True, wet_rule=WET_RULE)
gaps, skipped = teammate_race_gaps(clean, min_laps=MIN_LAPS, robust=ROBUST)
summary = race_season_summary(gaps)

n_races = laps.groupby(["Year", "Round"]).ngroups
print(f"{len(clean):,} clean dry laps of {len(laps):,} | {n_races} races, "
      f"{len(wet)} excluded as wet, {len(skipped)} skipped | "
      f"{len(gaps) // 2} teammate comparisons | median gap SE {gaps['GapSE'].median():.3f}s")

144,541 clean dry laps of 189,509 | 173 races, 20 excluded as wet, 20 skipped | 1360 teammate comparisons | median gap SE 0.103s


In [21]:
# @title Races excluded as wet
wet

,Year,Round,Event,WetTyreLaps,RainLaps
0,2018,11,German Grand Prix,35,264
1,2019,11,German Grand Prix,762,1059
2,2020,3,Hungarian Grand Prix,56,0
3,2020,14,Turkish Grand Prix,1077,0
4,2021,2,Emilia Romagna Grand Prix,498,337
5,2021,11,Hungarian Grand Prix,51,51
6,2021,15,Russian Grand Prix,67,100
7,2021,16,Turkish Grand Prix,1146,0
8,2022,4,Emilia Romagna Grand Prix,329,0
9,2022,7,Monaco Grand Prix,398,483


In [22]:
# @title Races skipped by the model (and why)
skipped

,Year,Round,Event,Driver,Reason
0,2018,14,Italian Grand Prix,MAG,only driver on MEDIUM: 47 laps dropped
1,2019,2,Bahrain Grand Prix,SAI,only driver on HARD: 14 laps dropped
2,2020,5,70th Anniversary Grand Prix,HUL,only driver on SOFT: 7 laps dropped
3,2020,6,Spanish Grand Prix,ALB,only driver on HARD: 20 laps dropped
4,2021,7,French Grand Prix,STR,only driver on SOFT: 18 laps dropped
5,2021,14,Italian Grand Prix,MAZ,only driver on SOFT: 1 laps dropped
6,2022,6,Spanish Grand Prix,MAG,only driver on HARD: 34 laps dropped
7,2022,8,Azerbaijan Grand Prix,TSU,only driver on SOFT: 11 laps dropped
8,2022,19,United States Grand Prix,RUS,only driver on SOFT: 1 laps dropped
9,2022,21,São Paulo Grand Prix,LAT,only driver on HARD: 10 laps dropped


In [23]:
# @title Season race-pace head-to-head
YEAR = 2025  # @param {type:"integer"}

summary.loc[summary["Year"] == YEAR,
            ["Team", "Driver", "Teammate", "Races", "H2H", "MedianGapSec", "MedianGapPct"]]

,Team,Driver,Teammate,Races,H2H,MedianGapSec,MedianGapPct
176,Red Bull Racing,VER,TSU,18,18-0,-1.187,-1.281
177,Red Bull Racing,VER,LAW,1,1-0,-0.879,-0.900
178,Alpine,GAS,DOO,3,3-0,-0.321,-0.345
179,Mercedes,RUS,ANT,20,14-6,-0.230,-0.262
180,Ferrari,LEC,HAM,20,17-3,-0.208,-0.243
181,Aston Martin,ALO,STR,18,14-4,-0.205,-0.228
182,Racing Bulls,HAD,LAW,18,11-7,-0.185,-0.222
183,Haas F1 Team,BEA,OCO,20,13-7,-0.138,-0.176
184,Racing Bulls,TSU,HAD,1,1-0,-0.119,-0.122
185,Alpine,GAS,COL,15,9-6,-0.098,-0.118


In [24]:
# @title Median race-pace gap to teammate, by driver
season = summary[summary["Year"] == YEAR]
fig = px.bar(
    season.sort_values("MedianGapPct"),
    x="MedianGapPct", y="Driver", color="Team", orientation="h",
    hover_data=["Teammate", "H2H", "Races", "MedianGapSec"],
    labels={"MedianGapPct": "Median race-pace gap to teammate (%)"},
    title=f"{YEAR}: median race-pace gap to teammate (negative = faster)",
)
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(height=650, yaxis={"categoryorder": "total descending"})
fig.show()

In [25]:
# @title One team, race by race (bars = ±2 standard errors)
TEAM = "McLaren"  # @param {type:"string"}

teams = sorted(gaps.loc[gaps["Year"] == YEAR, "Team"].unique())
if TEAM not in teams:
    print(f"'{TEAM}' not found in {YEAR}. Pick one of:", teams)
else:
    tg = gaps[(gaps["Year"] == YEAR) & (gaps["Team"] == TEAM)].copy()
    tg["Err"] = 2 * tg["GapSEPct"]
    fig = px.scatter(
        tg, x="Round", y="GapPct", color="Driver", error_y="Err",
        hover_data=["Event", "Teammate", "GapSec", "GapSE", "Laps", "TeammateLaps"],
        labels={"GapPct": "Race-pace gap to teammate (%)"},
        title=f"{TEAM} {YEAR}: race-pace gap by round",
    )
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [26]:
# @title One driver across seasons
DRIVER = "ALO"  # @param {type:"string"}

d = summary[summary["Driver"] == DRIVER]
if d.empty:
    print(f"No data for '{DRIVER}'. Use the 3-letter code, e.g. HAM, VER, LEC.")
else:
    fig = px.line(
        d, x="Year", y="MedianGapPct", markers=True, text="Teammate",
        hover_data=["Team", "H2H", "Races"],
        labels={"MedianGapPct": "Median race-pace gap to teammate (%)"},
        title=f"{DRIVER}: median race-pace gap to teammate by season",
    )
    fig.update_traces(textposition="top center")
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [27]:
# @title Qualifying vs race pace, per driver (needs notebook 00's data)
from f1sim.ingest import load_quali
from f1sim.features import teammate_gaps, season_summary

quali = season_summary(teammate_gaps(load_quali(get_paths())))
both = summary.merge(
    quali[["Year", "Driver", "Teammate", "MedianGapPct"]],
    on=["Year", "Driver", "Teammate"], suffixes=("Race", "Quali"),
)
season = both[both["Year"] == YEAR]
fig = px.scatter(
    season, x="MedianGapPctQuali", y="MedianGapPctRace", text="Driver", color="Team",
    labels={"MedianGapPctQuali": "Quali gap to teammate (%)",
            "MedianGapPctRace": "Race-pace gap to teammate (%)"},
    title=f"{YEAR}: qualifying vs race pace (bottom-left = faster in both)",
)
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_dash="dot"); fig.add_vline(x=0, line_dash="dot")
fig.show()